# IV-ANN: Model BASE (4-input, paper) vs Model FINM (2-input)

Extends Section 4.3 of:
> Liu, S., Oosterlee, C. W., & Bohte, S. M. (2019).  
> *Pricing options and computing implied volatilities using neural networks.*  
> arXiv:1901.08943v2

---

## NN computation of implied vol given call prices. Two approaches:

Let $F = Se^{r\tau}$, $x = \log(K/F)$ (forward log-moneyness),
$w = \sigma\sqrt{\tau}$ (total implied vol).

### Model BASE &mdash; 4-input (paper)

The parameterisation from Liu, Oosterlee &amp; Bohte (2019).  
Network inputs and output:

- Input 1: $\log(\tilde V/K)$
- Input 2: $S/K$
- Input 3: $r$
- Input 4: $\tau$
- Output: $\sigma$

### Model FINM &mdash; 2-input

Exploits two exact symmetries of the Black-Scholes formula.

**Step 1 &mdash; Scale homogeneity.**  
The Black-Scholes formula is homogeneous of degree 1 in $(S, K)$,
and $r, \tau$ enter $d_1, d_2$ only through $F = Se^{r\tau}$.
Scaling out $S$ collapses four inputs to two:
$$w = g\!\bigl(x,\;V/S\bigr), \qquad x = \log(K/F)$$
(where $V/S$ may be replaced by $\log(V/S)$ when using a "gradient squash" technique.)

**Step 2 &mdash; Put-call symmetry.**  
Learning the implied vol for OTM calls is enough.

A call price at moneyness $x < 0$ (ITM) maps into a hypothetical OTM call at moneyness $-x > 0$, hypothetically using the same $w$.  This mapping can be calculated without knowing $w$:
$$
\text{ITM call price}(x,w) \longrightarrow \text{OTM call price}(-x,w)
$$
by using put-call parity, then put-call symmetry.

Network inputs and output:

- Input 1: $|x|$
- Input 2: $\log(\text{OTM call price}/S)$
- Output: $w = \sigma\sqrt{\tau}$, converted to $\sigma$ via $\hat\sigma = \hat w/\sqrt{\tau}$

## Training target and loss

Both models are trained, validated, and evaluated on the $\sigma$ scale.
For Model FINM the network predicts $w$, which is converted to
$\hat\sigma = \hat w/\sqrt{\tau}$ observation-by-observation before
the loss is computed. This is equivalent to a $1/\tau$-weighted MSE on $w$:
$$\mathcal{L}_{\text{FINM}} = \frac{1}{n}\sum_i\!
\left(\frac{w_i - \hat w_i}{\sqrt{\tau_i}}\right)^{\!2}
= \frac{1}{n}\sum_i(\sigma_i - \hat\sigma_i)^2$$
The loss functions and convergence curves of both models are directly comparable.


## Cell 1 - Imports


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from scipy.stats import norm as scipy_norm
from scipy.stats import qmc
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time, warnings
warnings.filterwarnings('ignore')

import scipy
print('PyTorch :', torch.__version__)
print('SciPy   :', scipy.__version__, ' (need >= 1.7 for scipy.stats.qmc)')
print('NumPy   :', np.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device  :', DEVICE,
      '  [GPU active]' if DEVICE.type == 'cuda'
      else '  [CPU -- Runtime > Change runtime type > T4 GPU for speed]')


## Cell 2 - Configuration


Paper-exact Model BASE: set `N_SAMPLES=1_000_000`, `N_EPOCHS=3000`, `N_NEURONS=400`.


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

N_SAMPLES  = 200_000    # paper: 1_000_000
TEST_FRAC  = 0.10
VAL_FRAC   = 0.10
MIN_TV     = 1e-7

# Network architecture -- same for both models (paper Table 2)
N_HIDDEN_LAYERS = 4     # paper: 4
N_NEURONS       = 200   # paper: 400

N_EPOCHS   = 1000       # paper: 3000
BATCH_SIZE = 1024
LR_MAX     = 1e-3
LR_MIN     = 1e-5

print('Architecture : %d x %d' % (N_HIDDEN_LAYERS, N_NEURONS))
print('N_SAMPLES=%d  N_EPOCHS=%d  BATCH_SIZE=%d' % (N_SAMPLES, N_EPOCHS, BATCH_SIZE))


## Cell 3 - Black-Scholes helpers

We work in the variables $x = \log(K/F)$ and $w = \sigma\sqrt{\tau}$


In [ ]:
def bs_call_std(S_over_K, tau, r, sigma):
    # Standard BS call / K.
    # tau > 0 and sigma > 0 guaranteed by construction (sampled from [0.05, 1]).
    sqrt_tau = np.sqrt(tau)
    d1 = (np.log(S_over_K) + (r + 0.5*sigma**2)*tau) / (sigma*sqrt_tau)
    d2 = d1 - sigma*sqrt_tau
    return S_over_K*scipy_norm.cdf(d1) - np.exp(-r*tau)*scipy_norm.cdf(d2)


def bs_call_xw(x, w):
    # Problem 2a.  This is the "C" function.
    # It should return V/S as function of x=log(K/F) and w=sigma*sqrt(tau).
    # You may use the normal CDF, which has been imported as scipy_norm.cdf
    # FILL THIS IN


def bs_vega(S_over_K, tau, r, sigma):
    sqrt_tau = np.sqrt(tau)
    d1 = (np.log(S_over_K) + (r + 0.5*sigma**2)*tau) / (sigma*sqrt_tau)
    return S_over_K * scipy_norm.pdf(d1) * sqrt_tau


def intrinsic_over_K(S_over_K, tau, r):
    # max(S/K - e^{-r*tau}, 0)  -- intrinsic value / K
    return np.maximum(S_over_K - np.exp(-r*tau), 0.0)


def intrinsic_over_S(x):
    # max(1 - K/F, 0) = max(1 - exp(x), 0)  -- intrinsic value / S
    return np.maximum(1.0 - np.exp(x), 0.0)


def time_value_over_S(x, price_over_S):
    # Time value / S = call price / S - intrinsic / S.
    # Works for any x; caller provides the call price / S.
    return price_over_S - intrinsic_over_S(x)


def otm_price_given_itm_price_over_S(x, V_over_S):
    # Problem 2b.
    # Given x<0 (ITM) and V_over_S (which denotes call price / S),
    # Return C(-x,w), the (1/S-normalized) OTM call price at strike with moneyness -x > 0
    tv_over_S = time_value_over_S(x, V_over_S)    # This is TV/S, which = (put price)/S, because the call is ITM
    return         #FILL THIS IN


def otm_price_over_S(x, price_over_S):
    # OTM call price / S, always at |x| >= 0.
    # x >= 0: already OTM, zero intrinsic, so OTM price = call price / S
    # x <  0: ITM call; apply put-call symmetry to get OTM price at strike with moneyness -x
    return np.where(x >= 0,
                    price_over_S,
                    otm_price_given_itm_price_over_S(x, price_over_S))


# Verify the two BS formulations agree
_SK, _tau, _r, _sig = 1.05, 0.5, 0.04, 0.22
_x  = np.log(1.0/_SK) - _r*_tau
_w  = _sig * np.sqrt(_tau)
_p1 = bs_call_std(_SK, _tau, _r, _sig)
_p2 = bs_call_xw(_x, _w) * _SK
print('Standard V/K : %.8f' % _p1)
print('(x,w) form   : %.8f' % _p2)
print('Difference   : %.2e  =>  %s' % (abs(_p1-_p2), 'PASS' if abs(_p1-_p2)<1e-10 else 'FAIL'))


## Cell 4 - Verify both exact symmetries

### Symmetry 1: scale homogeneity
Two contracts with the same $(x, V/S)$ but different $(S, K, \tau, r)$
must yield the same $w = \sigma\sqrt{\tau}$.

### Symmetry 2: put-call symmetry

As discussed in HW7.2b


In [ ]:
print('=== Symmetry 1: scale homogeneity ===')
S1, K1, tau1, r1, sig1 = 100., 105., 0.5, 0.04, 0.22
F1  = S1*np.exp(r1*tau1);  x1 = np.log(K1/F1);  w1 = sig1*np.sqrt(tau1)
V1  = bs_call_std(S1/K1, tau1, r1, sig1) * K1
S2, tau2, r2 = 80., 1.2, 0.06
F2  = S2*np.exp(r2*tau2);  K2 = F2*np.exp(x1)
sig2 = w1/np.sqrt(tau2);   w2  = sig2*np.sqrt(tau2)
V2  = bs_call_std(S2/K2, tau2, r2, sig2) * K2
print('  x     : %.10f  vs  %.10f  diff=%.2e' % (x1, np.log(K2/F2), abs(x1-np.log(K2/F2))))
print('  V/S   : %.10f  vs  %.10f  diff=%.2e' % (V1/S1, V2/S2, abs(V1/S1-V2/S2)))
print('  w     : %.10f  vs  %.10f  diff=%.2e' % (w1, w2, abs(w1-w2)))
print('  Result: %s' % ('PASS' if abs(w1-w2)<1e-10 else 'FAIL'))

print()
print('=== Symmetry 2: put-call symmetry ===')
print('  %-6s %-5s  %-14s %-14s %-10s' % ('x', 'w', 'OTM direct', 'via symmetry', 'diff'))
for xv in [-0.5, -0.3, -0.1, -0.05]:
    for wv in [0.2, 0.5, 0.8]:
        itm_price = bs_call_xw(xv, wv)          # ITM call price / S
        direct    = bs_call_xw(-xv, wv)          # OTM call price / S (direct)
        via_sym   = otm_price_given_itm_price_over_S(xv, itm_price)   # via symmetry
        diff      = abs(direct - via_sym)
        print('  %-6.2f %-5.1f  %-14.8f %-14.8f %-10.2e %s'
              % (xv, wv, direct, via_sym, diff, 'PASS' if diff<1e-12 else 'FAIL'))


## Cell 5 - End of FINM 32000 requirements

If you want to do only what's required for FINM 32000, you can delete everything below here (and also Cell 2 and the PyTorch imports in Cell 1).

## Cell 6 - Data generation

Samples are drawn with `scipy.stats.qmc.LatinHypercube`.

**Model BASE (4-input):**
- Features: $\log(\tilde V/K)$, $S/K$, $r$, $\tau$
- Target: $\sigma$ (directly)

**Model FINM (2-input):**
- Features: $|x|$, $\log(\text{OTM call price}/S)$
- Target: $w = \sigma\sqrt{\tau}$, converted to $\sigma$ for loss
- An extra column $\sqrt{\tau}$ is appended to the feature matrix so the
  training loop can perform the conversion; it is never passed to the network.

ITM samples for Model FINM are **not discarded** -- put-call symmetry
maps them to OTM coordinates, so all samples contribute.

**Filter:** $\tilde V/K \geq 10^{-7}$, matching the paper's criterion (Section 4.3).


In [ ]:
PARAM_RANGES = {
    'S_over_K': (0.5, 1.4),
    'tau':      (0.05, 1.0),
    'r':        (0.0, 0.1),
    'sigma':    (0.05, 1.0),
}


def generate_dataset(n_samples=N_SAMPLES, seed=SEED, verbose=True):
    # Returns X_BASE, X_FINM, y_sigma, raw dict.
    #
    # X_BASE : float32 (N,4)  {log(TV/K), S/K, r, tau}
    # X_FINM : float32 (N,3)  {|x|, log(OTM/S), sqrt(tau)}
    #   -- col 2 carries sqrt(tau) for the loss function only;
    #      the network receives only cols 0 and 1.
    # y_sigma: float32 (N,)   sigma  -- shared training target
    #
    # Filter: TV_over_K >= MIN_TV, matching the paper's criterion (sec 4.3).

    if verbose:
        print('Generating %d samples via Latin-Hypercube Sampling ...' % n_samples)
    t0 = time.time()

    lo = np.array([v[0] for v in PARAM_RANGES.values()])
    hi = np.array([v[1] for v in PARAM_RANGES.values()])
    sampler = qmc.LatinHypercube(d=4, seed=seed)
    params  = qmc.scale(sampler.random(n=n_samples), lo, hi)

    S_K, tau, r, sigma = params[:,0], params[:,1], params[:,2], params[:,3]

    # Standard quantities for Model BASE
    V_over_K  = bs_call_std(S_K, tau, r, sigma)
    TV_over_K = V_over_K - intrinsic_over_K(S_K, tau, r)

    # Paper's filter: discard samples where TV_over_K < MIN_TV
    ok = TV_over_K >= MIN_TV
    def f(a): return a[ok]
    S_K, tau, r, sigma = f(S_K), f(tau), f(r), f(sigma)
    TV_over_K          = f(TV_over_K)
    n_kept = ok.sum()

    # Reduced-form quantities for Model FINM
    x        = np.log(1.0/S_K) - r*tau      # log(K/F)
    w        = sigma * np.sqrt(tau)
    sqrt_tau = np.sqrt(tau)
    V_over_S = bs_call_std(S_K, tau, r, sigma) / S_K   # call price / S

    # OTM price / S via put-call symmetry.
    # Pass the call price / S directly -- otm_price_given_itm_price_over_S does not
    # need w, making it usable on market data or test sets where w is unknown.
    otm_over_S = otm_price_over_S(x, V_over_S)

    # Safety check: confirm FINM log-inputs are positive after the paper's filter.
    bad = otm_over_S <= 0
    if bad.any():
        import warnings
        warnings.warn('%d samples have non-positive otm_over_S after filter -- '
                      'these will produce -inf in log.' % bad.sum())

    if verbose:
        print('  Kept %d/%d (%.1f%%) after TV_over_K >= %.0e filter'
              % (n_kept, n_samples, 100.*n_kept/n_samples, MIN_TV))
        print('  ITM fraction (x<0): %.1f%%  (mapped to OTM via put-call symmetry)'
              % (100.*(x<0).mean()))
        print('  Safety check -- non-positive otm_over_S: %d' % bad.sum())
        print('  Elapsed: %.1fs' % (time.time()-t0))

    # Model BASE features: {log(TV/K), S/K, r, tau}
    X_BASE  = np.column_stack([np.log(TV_over_K), S_K, r, tau]).astype(np.float32)

    # Model FINM features: {|x|, log(OTM/S), sqrt(tau)}
    X_FINM  = np.column_stack(
        [np.abs(x), np.log(np.maximum(otm_over_S, 1e-300)), sqrt_tau]
    ).astype(np.float32)

    y_sigma = sigma.astype(np.float32)

    if verbose:
        print('Model BASE : X=%s  sigma in [%.3f, %.3f]'
              % (X_BASE.shape, y_sigma.min(), y_sigma.max()))
        print('Model FINM : X=%s  |x| in [%.3f, %.3f]  log(OTM/S) in [%.2f, %.2f]'
              % (X_FINM.shape, X_FINM[:,0].min(), X_FINM[:,0].max(),
                 X_FINM[:,1].min(), X_FINM[:,1].max()))
        print('  (col 2 of X_FINM = sqrt(tau), used only in loss, not passed to network)')

    raw = dict(S_K=S_K, tau=tau, r=r, sigma=sigma, x=x, w=w)
    return X_BASE, X_FINM, y_sigma, raw


X_BASE, X_FINM, y_sigma, raw = generate_dataset(N_SAMPLES)


## Cell 7 - Feature distributions


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Model BASE (4 inputs)
for j, name in enumerate(['log(TV/K)', 'S/K', 'r', 'tau']):
    axes[0,j].hist(X_BASE[:,j], bins=70, color='#2176AE', alpha=0.82, density=True)
    axes[0,j].set_title('BASE: '+name, fontsize=12)
    axes[0,j].grid(True, ls='--', alpha=0.35)
axes[0,0].set_ylabel('Model BASE  (4-input)\nDensity', fontsize=11)

# Model FINM (2 inputs shown; col 2 = sqrt(tau) is auxiliary)
FINM_data  = [X_FINM[:,0], X_FINM[:,1], y_sigma, X_FINM[:,2]]
FINM_names = ['|x| = |log(K/F)|', 'log(OTM/S)',
              'sigma (shared target)', 'sqrt(tau)  [loss auxiliary]']
for j in range(4):
    col   = '#C1292E' if j < 2 else '#888888'
    alpha = 0.82      if j < 2 else 0.55
    axes[1,j].hist(FINM_data[j], bins=70, color=col, alpha=alpha, density=True)
    axes[1,j].set_title(FINM_names[j], fontsize=11)
    axes[1,j].grid(True, ls='--', alpha=0.35)
axes[1,0].set_ylabel('Model FINM  (2-input, OTM)\nDensity', fontsize=11)

fig.suptitle('Feature distributions: Model BASE (4-input) vs Model FINM (2-input, OTM)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


## Cell 8 - DataLoaders

Both models share the same train/val/test split indices.
Model FINM's loader carries 3 columns in `X`; the network
receives only the first 2, and column 2 ($\sqrt{\tau}$) is
used by the loss function to convert $\hat w \to \hat\sigma$.


In [ ]:
def make_loaders(X, y, test_frac=TEST_FRAC, val_frac=VAL_FRAC,
                 batch_size=BATCH_SIZE, seed=SEED):
    Xt, yt = torch.tensor(X), torch.tensor(y)
    ds = TensorDataset(Xt, yt)
    n  = len(ds)
    n_test = int(n * test_frac)
    n_tv   = n - n_test
    n_val  = int(n_tv * val_frac)
    n_tr   = n_tv - n_val
    g1 = torch.Generator().manual_seed(seed)
    g2 = torch.Generator().manual_seed(seed + 1)
    tv_ds, test_ds = random_split(ds,    [n_tv, n_test], generator=g1)
    tr_ds,  val_ds = random_split(tv_ds, [n_tr, n_val],  generator=g2)
    pin = DEVICE.type == 'cuda'
    return {
        'train': DataLoader(tr_ds,   batch_size=batch_size,   shuffle=True,
                            pin_memory=pin, num_workers=0),
        'val':   DataLoader(val_ds,  batch_size=batch_size*4, shuffle=False,
                            pin_memory=pin, num_workers=0),
        'test':  DataLoader(test_ds, batch_size=batch_size*4, shuffle=False,
                            pin_memory=pin, num_workers=0),
    }, dict(train=n_tr, val=n_val, test=n_test)


loaders_BASE, sizes = make_loaders(X_BASE, y_sigma)
loaders_FINM, _     = make_loaders(X_FINM, y_sigma)

print('Data split (shared):')
for k, v in sizes.items():
    print('  %-5s: %8d' % (k, v))


## Cell 9 - Architecture and training

Both models use the same MLP structure (Table 2 of the paper).
The only architectural difference is the input layer: 4 wide for BASE,
2 wide for FINM (the $\sqrt{\tau}$ column is never passed to the network).

**Loss function for Model FINM.**  
The network predicts $\hat w$.  Before computing MSE we convert:
$$\hat\sigma_i = \hat w_i / \sqrt{\tau_i}$$
using $\sqrt{\tau_i}$ from column 2 of the batch.  The MSE is then
computed on $\sigma$, making it directly comparable to Model BASE's loss.
This is equivalent to training FINM with a $1/\tau$-weighted MSE on $w$.


In [ ]:
class IVANN(nn.Module):
    def __init__(self, n_in, n_hidden=N_NEURONS, n_layers=N_HIDDEN_LAYERS):
        super().__init__()
        layers, in_dim = [], n_in
        for _ in range(n_layers):
            lin = nn.Linear(in_dim, n_hidden)
            nn.init.xavier_uniform_(lin.weight)
            nn.init.zeros_(lin.bias)
            layers.extend([lin, nn.ReLU()])
            in_dim = n_hidden
        out = nn.Linear(n_hidden, 1)
        nn.init.xavier_uniform_(out.weight)
        nn.init.zeros_(out.bias)
        layers.append(out)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def sigma_mse_loss(w_pred, X_batch, sigma_true, finm=False):
    # For BASE: w_pred is already sigma_pred (network outputs sigma directly).
    # For FINM: w_pred is the network output; divide by sqrt(tau) (col 2 of X)
    #           to convert to sigma before computing MSE.
    if finm:
        sqrt_tau   = X_batch[:, 2]       # auxiliary column, not a network input
        sigma_pred = w_pred / sqrt_tau
    else:
        sigma_pred = w_pred
    return torch.mean((sigma_pred - sigma_true)**2)


def compute_metrics(y_true, y_pred):
    # All metrics on the sigma scale.
    diff   = y_true - y_pred
    mse    = np.mean(diff**2)
    rmse   = np.sqrt(mse)
    mae    = np.mean(np.abs(diff))
    mape   = np.mean(np.abs(diff) / np.abs(y_true))
    ss_res = np.sum(diff**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    r2     = 1.0 - ss_res / ss_tot
    return dict(MSE=mse, RMSE=rmse, MAE=mae, MAPE=mape, R2=r2)


def predict_sigma(model, loader, finm=False):
    # Run inference; return (sigma_true, sigma_pred) both on the sigma scale.
    model.eval()
    yt_l, yp_l = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE)
            if finm:
                w_pred     = model(Xb[:, :2])      # network sees only cols 0,1
                sqrt_tau   = Xb[:, 2]
                sigma_pred = w_pred / sqrt_tau
            else:
                sigma_pred = model(Xb)
            yp_l.append(sigma_pred.cpu().numpy())
            yt_l.append(yb.numpy())
    return np.concatenate(yt_l), np.concatenate(yp_l)


def make_decay_scheduler(opt, lr_max, lr_min, n_epochs):
    log_ratio = float(np.log(lr_min / lr_max))
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda e: float(np.exp(log_ratio*e/max(n_epochs-1,1))))


def train_model(loaders, n_in, label, finm=False,
                n_epochs=N_EPOCHS, lr_max=LR_MAX, lr_min=LR_MIN):
    # n_in: number of features passed to the network (2 for FINM, 4 for BASE).
    # finm: if True, col 2 of each batch is sqrt(tau) used in the loss.
    model     = IVANN(n_in=n_in).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr_max)
    scheduler = make_decay_scheduler(optimiser, lr_max, lr_min, n_epochs)
    n_tr  = len(loaders['train'].dataset)
    n_val = len(loaders['val'].dataset)
    n_par = sum(p.numel() for p in model.parameters())

    print('='*65)
    print('  '+label)
    print('  %dx%d MLP  (%d params)  n_in=%d  train=%d  loss=sigma MSE'
          % (N_HIDDEN_LAYERS, N_NEURONS, n_par, n_in, n_tr))
    print('='*65)

    history    = dict(train_loss=[], val_loss=[])
    best_val   = float('inf')
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    t0 = time.time()

    for epoch in range(n_epochs):
        # ---- train ----
        model.train()
        run = 0.0
        for Xb, yb in loaders['train']:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            net_in = Xb[:, :2] if finm else Xb
            loss   = sigma_mse_loss(model(net_in), Xb, yb, finm=finm)
            loss.backward()
            optimiser.step()
            run += loss.item() * len(yb)
        train_loss = run / n_tr
        scheduler.step()

        # ---- validate ----
        model.eval()
        vrun = 0.0
        with torch.no_grad():
            for Xb, yb in loaders['val']:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                net_in = Xb[:, :2] if finm else Xb
                vrun  += sigma_mse_loss(model(net_in), Xb, yb, finm=finm).item() * len(yb)
        val_loss = vrun / n_val

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        every = max(1, n_epochs // 10)
        if (epoch+1) % every == 0 or epoch == 0:
            lr_now = scheduler.get_last_lr()[0]
            print('  Epoch %4d/%d  train=%.3e  val=%.3e  lr=%.2e  t=%.0fs'
                  % (epoch+1, n_epochs, train_loss, val_loss, lr_now, time.time()-t0))

    elapsed = time.time() - t0
    print('  Finished %.1fs  best val MSE (sigma scale)=%.3e' % (elapsed, best_val))
    model.load_state_dict(best_state)
    y_true, y_pred = predict_sigma(model, loaders['test'], finm=finm)
    m = compute_metrics(y_true, y_pred)
    print('  Test  MSE=%.3e  MAE=%.3e  MAPE=%.3e  R2=%.7f'
          % (m['MSE'], m['MAE'], m['MAPE'], m['R2']))
    print('-'*65)
    return model, history, m, y_true, y_pred, elapsed


n_par_base  = sum(p.numel() for p in IVANN(n_in=4).parameters())
n_par_finm  = sum(p.numel() for p in IVANN(n_in=2).parameters())
print('Model BASE : n_in=4  %dx%d: %d parameters' % (N_HIDDEN_LAYERS, N_NEURONS, n_par_base))
print('Model FINM : n_in=2  %dx%d: %d parameters' % (N_HIDDEN_LAYERS, N_NEURONS, n_par_finm))


## Cell 10 - Train Model BASE  (4-input, paper)


In [ ]:
model_BASE, hist_BASE, met_BASE, yt_BASE, yp_BASE, time_BASE = train_model(
    loaders_BASE, n_in=4, finm=False,
    label='Model BASE (4-input, paper): {log(TV/K), S/K, r, tau} -> sigma',
)


## Cell 11 - Train Model FINM  (2-input)


In [ ]:
model_FINM, hist_FINM, met_FINM, yt_FINM, yp_FINM, time_FINM = train_model(
    loaders_FINM, n_in=2, finm=True,
    label='Model FINM (2-input, OTM): {|x|, log(OTM/S)} -> w -> sigma',
)


## Cell 12 - Training dynamics

Both loss curves are on the **$\sigma$ scale** (MSE in $\sigma$ units),
so they are directly comparable. The convergence table below gives
the epoch at which each model first reaches a target val-RMSE,
from which training-time savings can be read off directly.


In [ ]:
ep = np.arange(1, N_EPOCHS+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
specs = [
    ('Model BASE  (4-input)', hist_BASE, '#2176AE', '-',   2.0),
    ('Model FINM  (2-input)', hist_FINM, '#C1292E', '--',  1.8),
]
for label, hist, col, ls, lw in specs:
    axes[0].plot(ep, np.log(hist['train_loss']), lw=lw, color=col, ls=ls, label=label)
    axes[1].plot(ep, np.log(hist['val_loss']),   lw=lw, color=col, ls=ls, label=label)

for ax, title in zip(axes, ['Training loss  (sigma MSE)', 'Validation loss  (sigma MSE)']):
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('log(MSE)  [sigma scale]', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, ls='--', alpha=0.35)

fig.suptitle('Training dynamics: both models on the sigma scale',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Convergence table ──────────────────────────────────────────────────────
rmse_BASE = np.sqrt(hist_BASE['val_loss'])
rmse_FINM = np.sqrt(hist_FINM['val_loss'])

def first_ep(arr, thr):
    hits = [i+1 for i, v in enumerate(arr) if v <= thr]
    return hits[0] if hits else None

# Estimate seconds per epoch from total training time
spe_BASE = time_BASE / N_EPOCHS
spe_FINM = time_FINM / N_EPOCHS

thresholds = [1e-2, 5e-3, 2e-3, 1e-3, 5e-4]
print('Convergence to target val-RMSE (sigma scale):')
print()
print('  %8s  %10s  %10s  %10s  %10s  %14s'
      % ('Target', 'BASE ep', 'BASE sec', 'FINM ep', 'FINM sec', 'Time saving'))
print('  '+'-'*68)
for thr in thresholds:
    eB = first_ep(rmse_BASE, thr)
    eF = first_ep(rmse_FINM, thr)
    sB = '%.1f' % (eB * spe_BASE) if eB else '>%.0f' % time_BASE
    sF = '%.1f' % (eF * spe_FINM) if eF else '>%.0f' % time_FINM
    if eB and eF:
        saving = (eB*spe_BASE - eF*spe_FINM)
        save_str = '%.1fs (%.1fx)' % (saving, eB*spe_BASE/max(eF*spe_FINM,1e-9))
    else:
        save_str = 'N/A'
    eB_str = str(eB) if eB else '>%d'%N_EPOCHS
    eF_str = str(eF) if eF else '>%d'%N_EPOCHS
    print('  %8.0e  %10s  %10s  %10s  %10s  %14s'
          % (thr, eB_str, sB, eF_str, sF, save_str))
print()
print('  BASE: %.1f s/epoch  |  FINM: %.1f s/epoch' % (spe_BASE, spe_FINM))


## Cell 13 - Summary comparison table  (test set)


In [ ]:
n_par_BASE = sum(p.numel() for p in model_BASE.parameters())
n_par_FINM = sum(p.numel() for p in model_FINM.parameters())

rows = [
    ('BASE  4-input (paper)', met_BASE, time_BASE, n_par_BASE),
    ('FINM  2-input',     met_FINM, time_FINM, n_par_FINM),
]

print()
print('='*80)
print('  PERFORMANCE COMPARISON  (sigma scale, held-out test set)')
print('='*80)
print('  %-24s  %7s  %9s  %9s  %9s  %9s  %7s'
      % ('Model', 'Params', 'MSE', 'MAE', 'MAPE', 'R2', 'Trn(s)'))
print('-'*80)
for name, m, t, np_ in rows:
    print('  %-24s  %7d  %9.3e  %9.3e  %9.3e  %9.7f  %7.0f'
          % (name, np_, m['MSE'], m['MAE'], m['MAPE'], m['R2'], t))
print('='*80)

print()
print('FINM vs BASE:')
print('  MAE  ratio (BASE/FINM): %.2fx  (%s)'
      % (met_BASE['MAE']/max(met_FINM['MAE'],1e-30),
         'FINM better' if met_FINM['MAE']<met_BASE['MAE'] else 'BASE better'))
print('  R2   BASE=%.7f  FINM=%.7f  delta=%+.2e'
      % (met_BASE['R2'], met_FINM['R2'], met_FINM['R2']-met_BASE['R2']))
print('  Full training time: BASE=%.0fs  FINM=%.0fs' % (time_BASE, time_FINM))


## Cell 14 - Out-of-sample performance  (test set, sigma scale)

All predictions and errors are on the $\sigma$ scale.
Model FINM predictions have already been converted from $\hat w$
to $\hat\sigma = \hat w / \sqrt{\tau}$ observation-by-observation.


In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

rng = np.random.default_rng(0)
idx = rng.choice(len(yt_BASE), size=min(60_000, len(yt_BASE)), replace=False)

model_specs = [
    ('Model BASE  (4-input, paper)', yt_BASE, yp_BASE, '#2176AE'),
    ('Model FINM  (2-input)',   yt_FINM, yp_FINM, '#C1292E'),
]

# Compute a common x-axis range for both error histograms
diff_BASE = yp_BASE - yt_BASE
diff_FINM = yp_FINM - yt_FINM
diff_all  = np.concatenate([diff_BASE, diff_FINM])
p_lo, p_hi = np.percentile(diff_all, [0.5, 99.5])   # 99% range, robust to outliers
hist_xlim  = (p_lo, p_hi)

for row, (label, yt, yp, color) in enumerate(model_specs):
    m = compute_metrics(yt, yp)

    ax1 = fig.add_subplot(gs[row, 0])
    ax1.scatter(yt[idx], yp[idx], s=0.5, alpha=0.22, color=color, rasterized=True)
    lo, hi = float(yt.min()), float(yt.max())
    ax1.plot([lo, hi], [lo, hi], 'k-', lw=1.4, label='Ideal')
    ax1.set_xlabel('Actual sigma', fontsize=11)
    ax1.set_ylabel('Predicted sigma', fontsize=11)
    ax1.set_title(label+'\nR2=%.7f' % m['R2'], fontsize=10)
    ax1.legend(fontsize=8)
    ax1.grid(True, ls='--', alpha=0.35)

    diff = yp - yt
    ax2  = fig.add_subplot(gs[row, 1])
    ax2b = ax2.twinx()
    ax2.hist(diff, bins=150, density=True, color=color, alpha=0.70,
             range=hist_xlim)
    sd = np.sort(diff)
    ax2b.plot(sd, np.linspace(0, 1, len(sd)), color='navy', lw=1.2)
    ax2.set_xlim(hist_xlim)
    ax2.set_xlabel('Predicted - Actual sigma', fontsize=11)
    ax2.set_ylabel('Density', fontsize=11)
    ax2b.set_ylabel('CDF', fontsize=11)
    ax2b.set_ylim(0, 1)
    ax2.set_title(label+'\nMAE=%.2e  MAPE=%.2e' % (m['MAE'], m['MAPE']), fontsize=10)
    ax2.grid(True, ls='--', alpha=0.3)

fig.suptitle('Out-of-sample performance on sigma scale  (test set)',
             fontsize=13, y=1.01)
plt.show()


## Cell 15 - Error by moneyness, maturity and volatility level  (test set)


In [ ]:
SK_list, tau_list = [], []
for Xb, _ in loaders_BASE['test']:
    SK_list.append(Xb[:,1].numpy())
    tau_list.append(Xb[:,3].numpy())
SK_test  = np.concatenate(SK_list)
tau_test = np.concatenate(tau_list)

ae_BASE = np.abs(yp_BASE - yt_BASE)
ae_FINM = np.abs(yp_FINM - yt_FINM)

sk_edges  = np.linspace(0.5,  1.4, 10)
tau_edges = np.linspace(0.05, 1.0, 10)

def bin2d(ae, x, y, xe, ye):
    nx, ny = len(xe)-1, len(ye)-1
    g = np.full((ny, nx), np.nan)
    xi = np.clip(np.digitize(x, xe)-1, 0, nx-1)
    yi = np.clip(np.digitize(y, ye)-1, 0, ny-1)
    for i in range(nx):
        for j in range(ny):
            m = (xi==i) & (yi==j)
            if m.sum() > 5:
                g[j,i] = ae[m].mean()
    return g

gB   = bin2d(ae_BASE, SK_test, tau_test, sk_edges, tau_edges)
gF   = bin2d(ae_FINM, SK_test, tau_test, sk_edges, tau_edges)
vmax = np.nanmax([gB, gF])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, grid, title in zip(axes, [gB, gF],
                            ['Model BASE (4-input)', 'Model FINM (2-input)']):
    im = ax.pcolormesh(sk_edges, tau_edges, grid, vmin=0, vmax=vmax, cmap='YlOrRd')
    plt.colorbar(im, ax=ax, label='Mean |error| sigma')
    ax.set_xlabel('S/K', fontsize=11)
    ax.set_ylabel('tau', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(sk_edges[0],  sk_edges[-1])
    ax.set_ylim(tau_edges[0], tau_edges[-1])

fig.suptitle('Mean absolute error by moneyness and maturity  (test set, sigma scale)',
             fontsize=12)
plt.tight_layout()
plt.show()

# Error by volatility level
sig_edges = np.linspace(0.05, 1.0, 25)
sig_c     = 0.5*(sig_edges[:-1]+sig_edges[1:])

def binnedmae(ae, yt, edges):
    xi = np.digitize(yt, edges)-1
    return [ae[xi==i].mean() if (xi==i).sum()>5 else np.nan
            for i in range(len(edges)-1)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(sig_c, binnedmae(ae_BASE, yt_BASE, sig_edges),
        color='#2176AE', lw=2, marker='o', ms=4, label='Model BASE (4-input)')
ax.plot(sig_c, binnedmae(ae_FINM, yt_FINM, sig_edges),
        color='#C1292E', lw=2, marker='s', ms=4, ls='--', label='Model FINM (2-input)')
ax.set_xlabel('Actual sigma', fontsize=12)
ax.set_ylabel('Mean |error| in sigma', fontsize=12)
ax.set_title('Error by volatility level  (test set)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, ls='--', alpha=0.35)
plt.tight_layout()
plt.show()


## Cell 16 - Summary and interpretation

**Both models train, validate, and report on the $\sigma$ scale.**  
Model FINM's network predicts $w = \sigma\sqrt{\tau}$, which is converted
to $\hat\sigma = \hat w/\sqrt{\tau}$ observation-by-observation before
the loss is computed.  This makes the loss functions directly comparable
and means the convergence curves in Cell 12 are a fair head-to-head race.

### The two exact symmetries used by Model FINM

**Symmetry 1 (scale homogeneity):** The Black-Scholes formula is
homogeneous of degree 1 in $(S, K)$, and $r, \tau$ enter $d_1, d_2$
only through $F = Se^{r\tau}$.  Scaling out $S$ and replacing
$(S/K, r, \tau)$ with $x = \log(K/F)$ reduces 4 inputs to 2.

**Symmetry 2 (put-call symmetry):**
This maps every ITM sample to an OTM domain representation,
so the network trains only on $|x| \geq 0$.

### Reading the convergence table (Cell 12)

The table gives, for each target val-RMSE threshold:
- The epoch at which each model first crosses that threshold
- The corresponding wall-clock time (epochs x seconds/epoch)
- The time saving (BASE time - FINM time) and speedup ratio

This is the cleanest way to convert accuracy improvements into
training-time savings: fix a target accuracy, measure how long
each model takes to reach it.

